# DISCO Treatment Effect Demo

This notebook shows how to run the DISCO algorithm directly using `disco.explain.disco_algorithm.DiscoRunner`.

Make sure the demo assets have been generated (see `experiments/run_demo.py`).


In [ ]:
# Add parent directory to path
import sys
import os
sys.path.append(os.path.abspath('../..'))


In [ ]:
from pathlib import Path
import json
import pickle
import numpy as np

from disco.explain.disco_algorithm import DiscoRunner
from disco.interventions.grids import Grids
from disco.interventions.tabular import FeatureAdd


def load_devices(devices_dir: Path):
    devices = []
    for pkl_path in sorted(Path(devices_dir).glob('device_*.pkl')):
        with pkl_path.open('rb') as f:
            devices.append(pickle.load(f))
    return devices


def load_probe_bundle(probes_dir: Path):
    probes_dir = Path(probes_dir)
    X_probe = np.load(probes_dir / 'X_probe.npy')
    meta_path = probes_dir / 'probe_meta.json'
    metadata = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    theta0 = np.array(metadata.get('theta0', []), dtype=float)
    trajectories = {}
    for npy_path in probes_dir.glob('*.npy'):
        if npy_path.name == 'X_probe.npy':
            continue
        parts = npy_path.stem.split('_', 2)
        if len(parts) < 3:
            continue
        device_id = '{}_{}'.format(parts[0], parts[1])
        interv_name = '_'.join(parts[2:])
        trajectories[(device_id, interv_name)] = np.load(npy_path)
    return X_probe, theta0, trajectories, metadata


def build_default_grids(theta0, n_post=8, gap=1.5):
    theta0 = np.asarray(theta0, dtype=float)
    base = float(theta0[-1]) if theta0.size else 0.0

    if theta0.size >= 2:
        diffs = np.diff(theta0)
        diffs = diffs[diffs > 0]
        if diffs.size:
            step = float(np.median(diffs))
        else:
            step = max(abs(base), 1.0) * 0.1
    else:
        scale = abs(base) if base != 0 else 1.0
        step = 0.1 * scale
    if step == 0:
        step = 0.5
    start = base + gap * step
    theta_max = start + step * max(n_post - 1, 1)
    theta = np.linspace(start, theta_max, n_post, dtype=float)
    return Grids(theta0=theta0, theta=theta)


runner = DiscoRunner()


In [ ]:
DEVICES_DIR = (Path('..').resolve() / 'demo_devices')
PROBES_DIR = (Path('..').resolve() / 'results' / 'demo_run' / 'probes')

devices = load_devices(DEVICES_DIR)
X_probe, theta0, trajectories, probe_meta = load_probe_bundle(PROBES_DIR)
available = sorted({name for _, name in trajectories.keys()})
print('Loaded {} devices'.format(len(devices)))
print('Probe set shape: {}'.format(X_probe.shape))
print('Available interventions: {}'.format(available))


In [ ]:
target_device_index = 0  # pick which device to explain
query_idx = 0  # choose a probe sample as the private query

target_device = devices[target_device_index]
x_star = X_probe[query_idx:query_idx + 1]
target_class = 1 if getattr(target_device, 'task', 'classification') == 'classification' else 0
print('Using device index {} and probe #{}'.format(target_device_index, query_idx))
print('Target task: {}'.format(target_device.task))


In [ ]:
INTERVENTIONS = {
    'add_f0': FeatureAdd(feat_idx=0, delta_max=3.0, name='add_f0'),
    'add_f1': FeatureAdd(feat_idx=1, delta_max=2.5, name='add_f1'),
}

intervention_key = 'add_f0'
intervention = INTERVENTIONS[intervention_key]
print('Selected intervention: {}'.format(intervention.name))


In [ ]:
grids = build_default_grids(theta0, n_post=8, gap=1.5)
output = runner.run(
    devices=devices,
    target_idx=target_device_index,
    x_star=x_star,
    X_probe=X_probe,
    trajectories=trajectories,
    grids=grids,
    intervention=intervention,
    target_class=target_class,
    mode='p',
    lam=0.01,
)
te_output = output.te_output
result = {'te_output': te_output, 'weights': output.weights}
result.update(output.diagnostics)
print('Weight L1 sum: {:.3f}'.format(np.sum(output.weights)))
print('AUC(|tau|): {:.4f}'.format(np.trapz(np.abs(te_output.tau), te_output.theta)))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(te_output.theta, te_output.tau, marker='o')
ax.fill_between(te_output.theta, te_output.tau, alpha=0.2)
ax.set_xlabel('Intervention intensity (theta)')
ax.set_ylabel('Treatment effect')
ax.set_title('Treatment effect curve for {}'.format(intervention.name))
plt.show()
